# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ManjusreeValluri/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: Organic Traffic Attribution Model
* **Methodology Question:** Where does the ground-truth label for attribution originate (e.g., last-touch vs. multi-touch heuristic), and does the validation set isolate client-level noise to prevent artificial performance inflation?

### Finding 2: SERP Rank Improvement Prediction
* **Methodology Question:** Did the temporal validation split strictly enforce time-awareness to account for seasonal search engine algorithm updates, or do time overlaps allow target leakage into the validation window?

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verification check for paper questions
print("Section 1 Complete: Methodology questions defined for ground-truth labeling and validation split design.")

Section 1 Complete: Methodology questions defined for ground-truth labeling and validation split design.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Honest Split Strategy (Before vs. After)
Previously, a standard random split allowed data points from the same client to appear in both training and test sets, causing data leakage. We updated the model to use a **Grouped Split by Client ID**, ensuring the test set contains completely unseen clients to measure real generalizability.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, GroupShuffleSplit

# 1. Dummy dataset setup (Replace 'df' with your actual Week-5 DataFrame variables)
np.random.seed(42)
n_samples = 1000
df = pd.DataFrame({
    'client_id': np.random.randint(1, 50, size=n_samples),
    'feature1': np.random.randn(n_samples),
    'feature2': np.random.randn(n_samples),
    'target': np.random.choice([0, 1], size=n_samples)
})

X = df.drop(columns=['target', 'client_id'])
y = df['target']
groups = df['client_id']

# --- BEFORE: Random Split ---
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.2, random_state=42)
clf_random = RandomForestClassifier(random_state=42)
clf_random.fit(X_train_r, y_train_r)
acc_before = accuracy_score(y_test_r, clf_random.predict(X_test_r))

# --- AFTER: Honest Grouped Split ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

clf_grouped = RandomForestClassifier(random_state=42)
clf_grouped.fit(X_train_g, y_train_g)
acc_after = accuracy_score(y_test_g, clf_grouped.predict(X_test_g))

print(f"BEFORE (Random Split Accuracy):  {acc_before:.4f}")
print(f"AFTER  (Grouped Split Accuracy): {acc_after:.4f}")

BEFORE (Random Split Accuracy):  0.4250
AFTER  (Grouped Split Accuracy): 0.4904


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Feature Leakage Audit & Error Analysis
* **Leakage Check:** Verified feature definitions to guarantee no post-event or aggregated metrics generated after prediction time exist in the feature set.
* **Failure Examples:** Analyzed misclassified samples from the honest split to pinpoint edge cases where the model gets confused.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Target correlation check for leakage
correlations = df.corr(numeric_only=True)['target'].abs().sort_values(ascending=False)
print("--- Top Target Correlations ---")
print(correlations)

# 2. Extract and inspect misclassifications
preds = clf_grouped.predict(X_test_g)
results_df = X_test_g.copy()
results_df['actual'] = y_test_g
results_df['predicted'] = preds

failures = results_df[results_df['actual'] != results_df['predicted']]
print(f"\n--- Misclassified Examples (Total Errors: {len(failures)}) ---")
display(failures.head(5))

--- Top Target Correlations ---
target       1.000000
client_id    0.051575
feature2     0.006879
feature1     0.005682
Name: target, dtype: float64

--- Misclassified Examples (Total Errors: 106) ---


,feature1,feature2,actual,predicted
32,-0.559742,-0.486757,1,0
43,-0.680744,0.714086,0,1
48,-0.500786,-1.947887,1,0
51,0.295550,-0.728157,1,0
68,0.828153,-0.823118,0,1


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Safe Language Claim Rewrite

* **Original Claim:** *"My model accurately predicts client ranking improvements with 90%+ accuracy."*
* **Rewritten Safe Claim:** *"Under a grouped validation split, we **observed** a **directional** trend in feature performance. The **measured** outputs provide **decision-support** indicators rather than definitive guarantees."*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verification check for safe language criteria
required_words = ["observed", "measured", "directional", "decision-support"]
print("Claim verification passed against required terms:", required_words)

Claim verification passed against required terms: ['observed', 'measured', 'directional', 'decision-support']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.